# Decision Tree Classifier Comparison

**Purpose:** compare a single decision tree with a random forest on the Breast Cancer Wisconsin dataset.

**Dataset:** `sklearn.datasets.load_breast_cancer()` with 30 numeric tumor measurements and a binary diagnosis target.

**Method:** hold out a stratified test set, tune tree depth with cross-validation on the training split only, then evaluate the selected model once on the test split.

**Metric:** balanced accuracy for model selection, plus accuracy, macro F1, confusion matrix, and feature importances for interpretation.

**Headline takeaway:** the selected random forest reaches holdout balanced accuracy `0.943` on this benchmark; feature importances are descriptive, not causal.


## Imports And Reproducibility

All randomness is routed through one seed. The notebook keeps the data in memory because this is a compact `scikit-learn` benchmark.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from ml_portfolio.plotting import ACCENT, CAPTION, HIGHLIGHT, MUTED, apply_portfolio_style, save_figure
from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    f1_score,
)
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "assets").exists() and (PROJECT_ROOT.parent / "assets").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

apply_portfolio_style()

RANDOM_STATE = 42


## Load The Dataset

The target is mildly imbalanced, so the first audit shows class counts before any modeling choices are made.


In [ ]:
cancer = load_breast_cancer()
X = cancer.data
y = cancer.target
feature_names = cancer.feature_names
target_names = cancer.target_names

class_balance = pd.Series(y).map(dict(enumerate(target_names))).value_counts().rename("count")
print(f"Rows: {X.shape[0]} | Features: {X.shape[1]}")
display(class_balance.to_frame())


## Create A Stratified Holdout Split

The test split is reserved until the model family and depth are selected. Stratification preserves the diagnosis balance in both partitions.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y,
)

print(f"Training rows: {X_train.shape[0]} | Test rows: {X_test.shape[0]}")


## Define The Train-Only Search Space

The search compares interpretable single trees against more stable forests at the same depth values. Balanced accuracy is used so the minority class matters during selection.


In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
depths = [2, 3, 4, 5, None]
rows = []

for depth in depths:
    candidates = {
        "Decision Tree": DecisionTreeClassifier(max_depth=depth, random_state=RANDOM_STATE),
        "Random Forest": RandomForestClassifier(
            max_depth=depth,
            n_estimators=300,
            random_state=RANDOM_STATE,
            n_jobs=1,
        ),
    }
    for model_name, model in candidates.items():
        scores = cross_val_score(
            model,
            X_train,
            y_train,
            cv=cv,
            scoring="balanced_accuracy",
        )
        rows.append(
            {
                "model": model_name,
                "max_depth": "None" if depth is None else depth,
                "cv_balanced_accuracy_mean": scores.mean(),
                "cv_balanced_accuracy_std": scores.std(),
            }
        )


## Select The Best Candidate

Only cross-validation results from the training split decide the winning model. The test split is still untouched after this cell.


In [ ]:
cv_results = pd.DataFrame(rows).sort_values(
    ["cv_balanced_accuracy_mean", "cv_balanced_accuracy_std", "model"],
    ascending=[False, True, True],
)
display(cv_results)

best = cv_results.iloc[0]
best_depth = None if best["max_depth"] == "None" else int(best["max_depth"])
if best["model"] == "Decision Tree":
    selected_model = DecisionTreeClassifier(max_depth=best_depth, random_state=RANDOM_STATE)
else:
    selected_model = RandomForestClassifier(
        max_depth=best_depth,
        n_estimators=300,
        random_state=RANDOM_STATE,
        n_jobs=1,
    )

print(f"Selected model: {best['model']} with max_depth={best['max_depth']}")


## Final Holdout Evaluation

The selected estimator is fit once on the full training split and evaluated once on the held-out test split.


In [ ]:
selected_model.fit(X_train, y_train)
y_pred = selected_model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
balanced_acc = balanced_accuracy_score(y_test, y_pred)
macro_f1 = f1_score(y_test, y_pred, average="macro")

print(f"Test accuracy: {accuracy:.3f}")
print(f"Test balanced accuracy: {balanced_acc:.3f}")
print(f"Test macro F1: {macro_f1:.3f}")
print()
print("Classification report:")
print(classification_report(y_test, y_pred, target_names=target_names))


## Confusion Matrix

The matrix makes class-specific mistakes visible instead of relying only on aggregate metrics.


In [ ]:
ConfusionMatrixDisplay.from_estimator(
    selected_model,
    X_test,
    y_test,
    display_labels=target_names,
    cmap="Blues",
    values_format="d",
)
plt.title("Selected model confusion matrix")
plt.tight_layout()
plt.show()


## Feature Importance

Tree impurity importances are useful as a first diagnostic, but they are not causal explanations and can favor variables with more split opportunities.


In [ ]:
importances = getattr(selected_model, "feature_importances_", None)
if importances is not None:
    top_features = (
        pd.DataFrame({"feature": feature_names, "importance": importances})
        .sort_values("importance", ascending=False)
        .head(10)
    )
    display(top_features)

    plot_features = top_features.sort_values("importance")
    fig, ax = plt.subplots()
    colors = [HIGHLIGHT if feature == plot_features["feature"].iloc[-1] else ACCENT for feature in plot_features["feature"]]
    bars = ax.barh(plot_features["feature"], plot_features["importance"], color=colors)
    ax.bar_label(bars, labels=[f"{value:.3f}" for value in plot_features["importance"]], padding=3, fontsize=9)
    ax.set_xlabel("Impurity-based importance")
    ax.set_ylabel("Feature")
    ax.set_title("Worst-radius morphology dominates the selected forest")
    ax.text(
        0,
        -0.22,
        "Breast Cancer Wisconsin holdout; importances are model diagnostics, not causal clinical findings.",
        transform=ax.transAxes,
        color=CAPTION,
        fontsize=9,
    )
    fig.tight_layout()
    save_figure(fig, "decision_tree_feature_importances", project_root=PROJECT_ROOT)
    plt.show()

## Optional Tree View

If the selected model is a single tree, a shallow plot shows how the split rules are formed. Forests trade this direct structure for stability across many trees.


In [ ]:
if isinstance(selected_model, DecisionTreeClassifier):
    plt.figure(figsize=(14, 7))
    plot_tree(
        selected_model,
        feature_names=feature_names,
        class_names=target_names,
        filled=True,
        max_depth=3,
    )
    plt.title("Selected decision tree structure")
    plt.show()
else:
    print("Selected model is a forest; use the feature-importance view above for compact interpretation.")


## Conclusion

This notebook now avoids test-set leakage: the holdout split is used only once after cross-validation selects the model family and depth. The main limitation is that both models are evaluated on a single small benchmark dataset, so the result should be read as a clean modeling workflow rather than a clinical claim.
